# Syntax: write a phrase structure grammar

In this notebook you write **phrase structure rules** and a **lexicon** in NLTK's format.
The computer then draws every tree your grammar gives for a sentence.

The program only shows you the tree(s). Deciding whether a tree is *right* is your job:
for each sentence, use **constituency tests** to argue for your structure.

**Before you start:** click **File → Save a copy in Drive**, and work in your copy.
Then run the **Setup** cell below (click ▶ on its left).

In [ ]:
# @title Setup: run this cell first (click ▶)
!pip install -q nltk svgling

import itertools
import nltk
import svgling
from nltk.grammar import Nonterminal
from IPython.display import display

MAX_TREES = 6


def _read_grammar(text):
    try:
        return nltk.CFG.fromstring(text)
    except ValueError as e:
        print("❌ Your grammar could not be read. Python says:")
        print("   ", e)
        print("Check: one rule per line, an arrow ->, alternatives separated by |, "
              "and every word in quotes, e.g.  N -> 'dog' | 'cat'")
        return None


def _words(sentence):
    return sentence.strip().rstrip(".!?").split()


def _warnings(g):
    if str(g.start()) != "S":
        print(f"⚠️  Your first rule starts with {g.start()}, so NLTK treats {g.start()} as the "
              "top of the tree. Put the S rule first.")
    lhs = {p.lhs() for p in g.productions()}
    undefined = sorted({str(s) for p in g.productions() for s in p.rhs()
                        if isinstance(s, Nonterminal) and s not in lhs})
    if undefined:
        print("⚠️  These symbols appear on the right of an arrow but never on the left: "
              + ", ".join(undefined))
        print("   If they are words, put them in quotes ('book'). "
              "If they are categories, add a rule for them.")
    loops = sorted({str(p.lhs()) for p in g.productions() if p.rhs() == (p.lhs(),)})
    if loops:
        print("⚠️  A rule like " + ", ".join(f"{x} -> {x}" for x in loops)
              + " lets the computer stack the same node forever. Remove it.")


def _phrases_built(g, words):
    """Phrases (2+ words) the grammar can build, to help find where a parse breaks."""
    chart = nltk.ChartParser(g).chart_parse(words)
    found = set()
    for e in chart.edges():
        if e.is_complete() and isinstance(e.lhs(), Nonterminal) and e.length() >= 2:
            found.add((e.start(), e.end(), str(e.lhs())))
    return sorted(found, key=lambda x: (x[0], -x[1]))


def show(grammar, sentence, max_trees=MAX_TREES):
    """Parse `sentence` with your grammar and draw every tree it gives."""
    g = _read_grammar(grammar)
    if g is None:
        return
    words = _words(sentence)
    print("Sentence:", " ".join(words))
    _warnings(g)
    lexicon = {s for p in g.productions() for s in p.rhs() if isinstance(s, str)}
    missing = [w for w in words if w not in lexicon]
    if missing:
        print("❌ These words are not in your lexicon:", ", ".join(missing))
        print("   Capital letters count: 'John' and 'john' are different words.")
        return
    trees = list(itertools.islice(nltk.ChartParser(g).parse(words), max_trees + 1))
    if not trees:
        print("❌ Your grammar does not generate this sentence.")
        built = _phrases_built(g, words)
        if built:
            print("   Phrases your grammar can build:")
            for i, j, cat in built:
                print(f"     {cat}: {' '.join(words[i:j])}")
            print("   Look for the part of the sentence that no rule puts together.")
        return
    if len(trees) > max_trees:
        print(f"Your grammar gives more than {max_trees} trees. Showing the first {max_trees}.")
        trees = trees[:max_trees]
    elif len(trees) == 1:
        print("✅ Your grammar gives 1 tree.")
    else:
        print(f"Your grammar gives {len(trees)} trees, so it is ambiguous for this sentence.")
        print("Use constituency tests to decide which tree matches the meaning.")
    for n, t in enumerate(trees, 1):
        print(f"\nTree {n}:  {t._pformat_flat('', '()', False)}")
        display(svgling.draw_tree(t))


def check(grammar, good, bad=()):
    """Count the trees for each sentence. Good sentences need 1+, starred ones need 0."""
    g = _read_grammar(grammar)
    if g is None:
        return
    _warnings(g)
    lexicon = {s for p in g.productions() for s in p.rhs() if isinstance(s, str)}
    parser = nltk.ChartParser(g)
    for sentence, should_parse in [(s, True) for s in good] + [(s, False) for s in bad]:
        words = _words(sentence)
        missing = [w for w in words if w not in lexicon]
        n = 0 if missing else len(list(itertools.islice(parser.parse(words), MAX_TREES + 1)))
        # 星號句缺詞時不算過關，不然空白文法也會全拿 ✅
        ok = not missing and (n > 0) == should_parse
        label = ("" if should_parse else "*") + " ".join(words)
        note = f"missing words: {', '.join(missing)}" if missing else f"{n} tree(s)"
        print(("✅ " if ok else "❌ ") + f"{label:<45} {note}")


print("Ready. Run the cells below in order.")


## How the grammar format works

Here is a grammar that generates *Mary saw Bob*. Run the cell to see its tree.

* One rule per line: `S -> NP VP` means "an S is an NP followed by a VP".
* `|` separates alternatives: `NP -> Det N | Det N PP`.
* Words go in quotes and must match the sentence exactly (`'John'` is not `'john'`).
* The **first** rule must be the `S` rule: it is the top of the tree.
* Lines that start with `#` are comments; the computer ignores them.
* Leave out the final period when you type the sentence.

In [ ]:
grammar0 = """
# Phrase structure rules
S -> NP VP
VP -> V NP | V NP PP
PP -> P NP
NP -> 'John' | 'Mary' | 'Bob' | Det N | Det N PP

# Lexicon
V -> 'saw' | 'ate' | 'walked'
Det -> 'a' | 'an' | 'the' | 'my'
N -> 'man' | 'dog' | 'cat' | 'telescope' | 'park'
P -> 'in' | 'on' | 'by' | 'with'
"""

show(grammar0, "Mary saw Bob")

## Constituency tests you can use

| Test | What you do | Example |
|---|---|---|
| Substitution (pro-form) | Replace the string with *it, she, they, there, then, do so* | *the girl with the hat* → *she* |
| Movement | Move the string to the front of the sentence | *On the desk, John put the book.* |
| Question / fragment answer | Ask a question; the string can be the short answer | *Where did John put the book?* — *On the desk.* |
| Coordination | Join the string with another of the same kind using *and* | *on the desk and under the chair* |

A string that passes these tests is a constituent, so it should be **one node** in your tree.
A string that fails them should *not* be one node.

## Task 1. *John put the book on the desk.*

Write the rules and the lexicon so that your grammar generates this sentence. *put* needs both an object and a location: *\*John put the book* is not a sentence.

In [ ]:
grammar1 = """
# Phrase structure rules
S -> NP VP

# Lexicon
"""

show(grammar1, "John put the book on the desk")

**Argue for your tree.** Is *the book* one constituent, or is *the book on the desk* one constituent? Give at least two tests. If your grammar gives two trees, say which one is right.

**Your argument** (double-click to edit)

My tree: *(which tree number, or describe it)*

Test 1:

Test 2:


## Task 2. *The girl with the hat smiled.*

Write a grammar for this sentence. *smiled* has no object.

In [ ]:
grammar2 = """
# Phrase structure rules
S -> NP VP

# Lexicon
"""

show(grammar2, "The girl with the hat smiled")

**Argue for your tree.** Show with a test that *the girl with the hat* is one NP.

**Your argument** (double-click to edit)

My tree: *(which tree number, or describe it)*

Test 1:

Test 2:


## Task 3. *Bob ate the cake with a fork.*

Write a grammar for this sentence. Think about what *with a fork* goes with: the cake, or the eating?

In [ ]:
grammar3 = """
# Phrase structure rules
S -> NP VP

# Lexicon
"""

show(grammar3, "Bob ate the cake with a fork")

**Argue for your tree.** Use a test to show whether *the cake with a fork* is one constituent or not.

**Your argument** (double-click to edit)

My tree: *(which tree number, or describe it)*

Test 1:

Test 2:


## Task 4. *Mary saw the man with the telescope.*

This sentence has two meanings. Write **one** grammar that gives **two** trees, one for each meaning.

In [ ]:
grammar4 = """
# Phrase structure rules
S -> NP VP

# Lexicon
"""

show(grammar4, "Mary saw the man with the telescope")

**Argue for your tree.** For each tree, write the meaning it gives, and one test that brings out that meaning.

**Your argument** (double-click to edit)

My tree: *(which tree number, or describe it)*

Test 1:

Test 2:


## Task 5. One grammar for everything

Combine your grammars into **one** grammar that generates all four sentences
and does **not** generate the starred ones. `check` prints ✅ when a good sentence gets
at least one tree and a starred sentence gets none.

Hint: *put*, *ate*, *saw* and *smiled* do not take the same things after them.
You may need more than one verb category (for example `Vt`, `Vi`, `Vput`).

In [ ]:
grammar5 = """
# Phrase structure rules
S -> NP VP

# Lexicon
"""

check(grammar5,
      good=['John put the book on the desk', 'The girl with the hat smiled', 'Bob ate the cake with a fork', 'Mary saw the man with the telescope'],
      bad=['John put the book', 'The girl smiled the cake', 'Mary saw'])